# Lab 10: Code Generation and Pair Programming with LangGraph

In this series of exercises, we will use LLMs to generate code based on specific requirements and then compare the output against predefined test cases to ensure correctness. The focus will be on generating functional code and executing tests to verify that it meets the given specifications.

After having experimented with a single-LLM approach, we will use the LangGraph framework to implement a multi-agent architecture composed by two LLMs that simulates pair-programming.

### Step 1: Requirements

The function receives the grades of a student on her courses (for simplicity exactly six grades for six courses are considered) and computes the average. Grades can be from 18 to 30, or 30Laude (which is represented by the int number 33). The average is computed excluding the best and worse grade.

In [ ]:


def compute_average(grades: list[float]) -> float:
    """
    Grades can be from 18 to 30, or 30Laude (which is represented by the int number 33).
    The average is computed excluding the best and worse grade.

    Args:
        grades (list): A list of six grades, where each grade is between 18 and 33.

    Returns:
        float: The computed average.
    """
    #TODO
    #define here your reference solution
    pass


# Example usage
grades = [18, 25, 30, 33, 22, 28]
average = compute_average(grades)
print(f"The computed average is: {average:.2f}")



gpt_result = """""" #TODO paste here the function produced when prompting ChatGPT

# The dictionary my_codes will contain all the generated codes. Let's start by adding the one generated by ChatGpt
my_codes = {}
my_codes["GPT"] = gpt_result


### Step 2: Test Cases

To generate a comprehensive black-box test suite for the compute_average function, we will create tests that cover different equivalence classes, boundary conditions, and typical scenarios.

- Valid inputs:
  - Standard valid grades within the range of 18 to 33, including "30Laude" (33).
  - Tests where all grades are different, with some being 30Laude.

- Boundary values:
  - Input with the lowest valid grade (18) and the highest valid grade (33).

- Invalid inputs:
  - A grade list that has more or fewer than 6 grades.
  - Grades outside the valid range (less than 18 or greater than 33).
  - non-numeric inputs

- Edge cases:
  - All grades are the same.
  - All grades are 30Laude (33).


In [ ]:
#----------GOOGLE COLAB ONLY-----------------
#!pip install pytest
#--------------------------------------------



import pytest

# An example of a valid test case. Notice the use of the assert function and the pytest.approx to cope with possible roundings

def test_valid_grades():
    grades = [18, 25, 30, 33, 22, 28]
    result = compute_average(grades)
    assert result == pytest.approx(26.25, rel=1e-2)

# An example of an invalid test case. Notice the use of the with pytest.raises syntax, to catch exceptions in the function

# Invalid test cases
def test_more_than_six_grades():
    grades = [18, 25, 30, 33, 22, 28, 29]
    with pytest.raises(ValueError): # HINT: pay attention to the type of exception expected
        compute_average(grades)

#TODO
# Try to define here additional valid and invalid test cases, that can be used to verify the generated code.
# Further examples are already defined in test_cases_01.py

### Step 3: Run the test cases

This code defines a function run_tests that uses the ipytest library to run test cases within a Jupyter notebook. The ipytest library seeks for test cases in the notebook and launches them. The '-vv' parameter is used to provide a verbose output.

In [ ]:

# Run the test suite

#----------GOOGLE COLAB ONLY-----------------
#!pip install ipytest
#--------------------------------------------


import ipytest
ipytest.autoconfig()

def run_tests():
    ipytest.run('-vv')

# Running the tests
run_tests()


### Step 4: Generating code with CodeLLAMA

In this step, we leverage Llama to automatically generate Python code based on specified requirements.

In this step, we define a prompt for coding, which includes the requirements, the arguments, and the expected returns.


In [ ]:
# ------------------ Initial setup ---------------------

import torch

# Detect the device available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ",
      device)  # You may need to execute "pip install torch --index-url https://download.pytorch.org/whl/cu124" (or other versions in place of the 12.4) in order to use the GPU

# HuggingFace login: it is needed only once

#----------------Google Colab only---------------------
# from huggingface_hub import login
#from google.colab import userdata
#login(userdata.get('HF_TOKEN'))
#----------------Google Colab only---------------------

#----------------Local Runtime only---------------------
import os
from dotenv import load_dotenv  # pip install python-dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.getenv("HF_TOKEN")  # Create this constant in the file .env (put it in the root dir of your project)
login(token=hf_token)
#----------------Local Runtime only---------------------

**ATTENTION**: The following function has been modified with respect to the previous labs. This is intended to make the integration of the text\_generation\_pipeline object easier within LangGraph (see the second part of the Lab). The comments in the code below highlight the changes made with respect to the previous labs.

In [ ]:
from transformers import AutoTokenizer
from transformers import BitsAndBytesConfig  # pip install bitsandbytes

from transformers import (
    AutoModelForCausalLM,
    GenerationConfig,
    pipeline,
    TextGenerationPipeline,
)


def load_model(model_id: str, quantization: bool = False) -> TextGenerationPipeline:
    """Loads a pretrained causal language model and builds a configured text-generation pipeline.

    Note: The initial execution may take significant time to download the model weights.

    Args:
        model_id (str): The Hugging Face model identifier or local path.
        quantization (bool, optional): Whether to use quantization. Defaults to False.

    Returns:
        TextGenerationPipeline: A Hugging Face text-generation pipeline initialized with
            the float16 causal LM, its tokenizer, and custom generation configuration.
    """
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    if quantization:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto",
                                                     quantization_config=quantization_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto", )
    tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)

    tokenizer.pad_token = tokenizer.eos_token

    # Configure generation parameters
    gen_config = GenerationConfig.from_model_config(model.config)
    gen_config.temperature = 0.1              # Regulate randomness, use with do_sample = True
    # gen_config.max_length = None              # Since max_new_tokens is already set
    #gen_config.repetition_penalty = 2.0  # Penalize repetition
    #gen_config.no_repeat_ngram_size = 3  # Avoid repeating trigrams (IMPORTANT: do not set this parameter if you target code generation, because code naturally repeats short token sequences)
    gen_config.do_sample = True  # Set to False in order to use Greedy or Beam search
    #gen_config.num_beams = 3  # Use with do_sample = False
    gen_config.eos_token_id = tokenizer.eos_token_id  # End generation at EOS token
    gen_config.pad_token_id = tokenizer.pad_token_id
    gen_config.early_stopping = True

    model.generation_config = gen_config

    text_generation_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text = False    # ATTENTION: as this parameter has been added, the syntax for obtaining the response text has changes (see below)
    )

    return text_generation_pipeline

## Usage example:
# text_generation_pipeline = load_model("causal-language-model")
# messages = [...]
# outputs = text_generation_pipeline(messages, max_new_tokens = 200)
# print(outputs[0]["generated_text"]) <- ** CHANGED SYNTAX **

In [ ]:

#TODO
# Define the prompts to ask the model to generate the target code
# HINT: Also include instructions about the output format necessary for the subsequent phases
sys_prompt = """
"""
query = """
"""

text_generation_pipeline = load_model("meta-llama/Llama-3.2-3B-Instruct", quantization=True) # Quantization is suggested. Indeed, the part of this lab that leverages LangGraph includes large prompts.

# Generate code using the model
generated_code = ""
my_codes["CODELLAMA"] = generated_code

## Organizing and analyzing test case results with pytest

The code below automates the process of running test cases in a Python file using pytest and parsing the results. It defines a function, parse_test_results, that extracts the counts of errors, failures, and passes from a summary line of the pytest output using regular expressions. The script runs the specified test file (test_cases_01.py) via a subprocess call to pytest, capturing its output in plain text. From the output, it locates the summary line containing test results and uses the parse_test_results function to extract key metrics. It calculates the functional correctness ratio as the fraction of passed tests to the total number of tests executed and prints the counts of passed, failed, and errored tests, along with the computed functional correctness ratio. This process provides a clear and automated way to assess the reliability of the tested code.

In [ ]:
import sys
from pathlib import Path
import pytest
import subprocess
import re

def parse_test_results(result_string):
    """
    Parses the test result string to extract numbers of errors, failures, and passes.

    Args:
        result_string (str): A string containing test results (e.g., "3 failed, 11 passed in 0.04s").

    Returns:
        dict: A dictionary with keys 'errors', 'failures', and 'passed' and their respective counts.
    """
    # Regular expressions to match the counts for errors, failures, and passes
    errors = re.search(r"(\d+)\s+errors?", result_string)
    failures = re.search(r"(\d+)\s+failed", result_string)
    passes = re.search(r"(\d+)\s+passed", result_string)

    # Extract numbers or default to 0 if not found
    return int(errors.group(1)) if errors else 0, int(failures.group(1)) if failures else 0, int(passes.group(1)) if passes else 0



# Define the path to your test file
test_file = Path("./test_cases_01.py").resolve()

with open('function_01.py', 'w') as file:
    file.write(my_codes["GPT"])

# Run the pytest command and capture the output
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "--disable-warnings", "--tb=short", "-q", "--color=no"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Extract test results from the pytest output
output_lines = "" # TODO write the code to extract and print all the output lines from the result of the call to the subprocess.
result_line = "" # TODO from the list of lines, identify the one that contains the overall results

errors, failures, passes = parse_test_results(result_line)
gpt_functional_correctness = passes/(errors+failures+passes)


# Print the results
print("GPT")
print(f"# Passed: {passes}")
print(f"# Failed: {failures}")
print(f"# Errors: {errors}")
print(f"Functional Correctness Ratio: {gpt_functional_correctness:.2f}")


### Step 6: comparing results

We now extend our analysis to compare the results provided by different agents.

For simplicity, we store the function always on the same file (e.g., function_01.py) and we re-execute the same test file (e.g., test_cases_01.py)

We evaluate the results based on the functional correctness ratio.

In [ ]:

with open('function_01.py', 'w') as file:
    file.write(my_codes["CODELLAMA"])


result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "--disable-warnings", "--tb=short", "-q", "--color=no"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Extract test results from the pytest output
output_lines = "" # TODO write the code to extract and print all the output lines from the result of the call to the subprocess.
result_line = "" # TODO from the list of lines, identify the one that contains the overall results

errors, failures, passes = parse_test_results(result_line)
llama_functional_correctness = passes/(errors+failures+passes)


# Print the results
print("CODELLAMA")
print(f"# Passed: {passes}")
print(f"# Failed: {failures}")
print(f"# Errors: {errors}")
print(f"Functional Correctness Ratio: {llama_functional_correctness:.2f}")



### Step 7: Analyzing and comparing the quality of the generated code

Static code quality metrics are used to evaluate the structure, readability, and maintainability of code without executing it. Below is a Python program that calculates some common static code quality metrics for a given Python file:

- Lines of Code (LOC): The total number of lines in the file.
- Comment Density: The percentage of lines that are comments.
- Cyclomatic Complexity: A measure of the complexity of a program based on the number of linearly independent paths.
- Maintainability Index (MI): a software metric used to predict how maintainable the code is. It’s often calculated using a combination of Cyclomatic Complexity (CC), Lines of Code (LOC), and Halstead Volume (HV).

The **Radon** tool is a Python package that helps analyze various aspects of code quality, such as Cyclomatic Complexity (CC), Maintainability Index (MI), Raw Metrics (LOC), and Halstead metrics. You can use Radon to analyze the static code quality of your Python code by running it from the command line or through Python code.

The tool can be installed by the following command: pip install radon


In [ ]:
# --------------------------GOOGLE COLAB ONLY----------------------------
#!pip install radon
#------------------------------------------------------------------------

import radon.complexity as radon_complexity
import radon.metrics as radon_metrics
import radon.raw as radon_raw

def analyze_code_with_radon(file_path):
    """
    Analyzes a Python file and calculates code quality metrics:
    - Cyclomatic Complexity (CC)
    - Maintainability Index (MI)
    - Raw Metrics (LOC, number of functions, etc.)

    Args:
        file_path (str): Path to the Python file to analyze.
    """
    with open(file_path, 'r') as file:
        code = file.read()

    # Cyclomatic Complexity Analysis (CC)
    cc_results = radon_complexity.cc_visit(code)
    print("Cyclomatic Complexity:")
    for result in cc_results:
        print(f"Function: {result.name}, Complexity: {result.complexity}")

    # Maintainability Index (MI)
    maintainability_index = radon_metrics.mi_visit(code, multi=False)  # Set multi=False for single file
    print(f"\nMaintainability Index: {maintainability_index}")

    # Raw Metrics (LOC, number of functions, etc.)
    raw_metrics = radon_raw.analyze(code)  # Use analyse method for raw metrics
    print(f"\nLines of Code (LOC): {raw_metrics.loc}")
    print(f"Number of Comments: {raw_metrics.comments}")
    #print(f"Blank Lines: {raw_metrics.blank_lines}")

    result = {}
    result["CC"] = cc_results[0].complexity
    result["MI"] = maintainability_index
    result["LOC"] = raw_metrics.loc
    result["Comments"] = raw_metrics.comments
    return result

print(my_codes)


overall_results = {}

#TODO: cycle over all the "my_codes" dictionary. Sequentially, save each code in the function_01.py file and:
# - execute the radon analysis with the function above
# - run the python tests by using the code provided in a previous cell
# Example of the final structure of the overall_results dictionary:
# {'GPT': {'CC': 4, 'MI': 93.26472421003476, 'LOC': 24, 'Comments': 3, 'FunctionalCorrectness': 0.80}, 'CODELLAMA': {'CC': 1, 'MI': 77.16231550361674, 'LOC': 5, 'Comments': 0, 'FunctionalCorrectness': 0.43}}


print(overall_results)



### Step 8: Plotting the metrics

Generate a set of bar charts to visualize the metrics stored in the overall_results dictionary, such as Cyclomatic Complexity (CC), Maintainability Index (MI), Lines of Code (LOC), and the number of comments.

Using Matplotlib, create a 3x2 grid of subplots, each one dedicated to a different metric.

In [ ]:
import matplotlib.pyplot as plt


languages = list(overall_results.keys())


cc_values = [overall_results[language]['CC'] for language in languages]
mi_values = [overall_results[language]['MI'] for language in languages]
loc_values = [overall_results[language]['LOC'] for language in languages]
comments_values = [overall_results[language]['Comments'] for language in languages]

# Create a figure with subplots for each metric
fig, axs = plt.subplots(3, 2, figsize=(10, 8))

# Plotting CC values
axs[0, 0].bar(languages, cc_values, color='skyblue')
axs[0, 0].set_title('Cyclomatic Complexity (CC)')
axs[0, 0].set_ylabel('CC')

# Plotting MI values
axs[0, 1].bar(languages, mi_values, color='lightgreen')
axs[0, 1].set_title('Maintainability Index (MI)')
axs[0, 1].set_ylabel('MI')

# Plotting LOC values
axs[1, 0].bar(languages, loc_values, color='blue')
axs[1, 0].set_title('Lines of Code (LOC)')
axs[1, 0].set_ylabel('LOC')

# Plotting Comments values
axs[1, 1].bar(languages, comments_values, color='lightcoral')
axs[1, 1].set_title('Number of Comments')
axs[1, 1].set_ylabel('Comments')

#plotting functional correctness
axs[2, 0].bar(languages, [gpt_functional_correctness, llama_functional_correctness], color='green')
axs[2, 0].set_title('Functional correctness')


# Adjust layout and display the plot
plt.tight_layout()
plt.show()


## A Multi-agent Architecture with LanGraph to Implement Pair-programming

**LangGraph** is a framework built on top of LangChain for creating stateful AI agents. It models workflows as cyclical graphs—allowing LLMs to loop, maintain memory, and use tools iteratively.

In this lab, we'll define two agents: Alice and Bob.

We will perform **test-driven development** by taking one test case at a time and writing the necessary code to make it to pass. The agents won't have access to the tests code, but they will only be able to read the output of the tests execution. By keeping this **black-box** approach, we prevent the agents to **overfit** the tests.

Before each modification to the code, they will have a short discussion, which will allow them to reason about how to proceed with code generation.

This cycle repeats until all tests pass. However, if a test fails for more than three consecutive reasoning-and-coding iterations, the agents will roll back to the last valid code state.

The iterative process is schematized below:

![langraph_process.jpg](langraph_process.jpg)



### Step 1: Define the inter-agents shared state

In LangGraph, a **shared state** is a central data structure that acts as the graph's collective memory. It is defined via a dedicated class.

The list of chat `messages` is wrapped in the `Annotated` LangGraph's class. This tells LangGraph to keep the whole history of states: a new value of this attribute won't replace the previous values, but will be appended instead.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class PairState(TypedDict):

    messages: Annotated[list[AnyMessage], add_messages]
    """Shared conversation history between Alice, Bob, and system test logs.
    Test logs are embedded within the conversation.

    NOTE: to avoid saturating the context window, only **summaries** of the test logs are saved."""

    current_test_idx: int
    """The current target test in the iterative test-driven development process."""

    # Codebase snapshots
    code: str
    last_known_good_code: str
    """Last version of the code that passed all the tests"""

    # Pre-coding discussion tracking
    discussion_turns: int
    """Counter of the messages exchanged between the two agents after the execution of the tests and before the code update."""
    ready_to_code: bool
    """Flag that indicates that the discussion prior to code modification has finished before the maximum number of discussion turns has been reached."""
    agreed_approach: str
    """Summary of the final decision resulted from the discussion."""

    # Role management (either "Alice" or "Bob" holds the keyboard)
    driver: Literal["Alice", "Bob"]
    """Indicates which agent will implement the agreed code modifications."""

    # Tests execution tracking
    failed_attempts: int
    """Counter that tracks consecutive test failures on the current target test. It resets back to 0 after a rollback or whenever the test suite passes."""
    last_tests_output: str
    """Output of the last execution of all the test suite up to  current_test_idx."""

### Step 2: Wrap the TextGenerationPipeline into a ChatModel

The TextGenerationPipeline is from the HuggingFace library.
In LangGraph (via LangChain), a **ChatModel** is the core LLM interface.

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

# Wrap for LangChain
base_llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline,
    pipeline_kwargs={
        "max_new_tokens": 1024, # Overrides the one defined in text_generation_pipeline
        #"return_full_text": False,  <- In this case, this is USELESS, as we already specified that we don't need the full prompt back when creating the text_generation_pipeline object
    },
)

# Wrap for LangGraph
llm = ChatHuggingFace(llm=base_llm)

### Step 3: Implement the Discussion Node

We need two building blocks:

---

#### 1. Pydantic-based structured communication
For this purpose, we define a Pydantic model named `DiscussionTurn`. This is a special class that defines the format of the outputs generated by the LLM during this discussion step.

**Pydantic** is a library for data validation using standard type annotations. It ensures that incoming data matches expected types at runtime, automatically parsing it or raising errors if it's invalid.

---

#### 2. LangChain's Message objects

In LangChain, conversations are modeled as an ordered list of structured `Message` objects rather than raw dictionaries containing text strings (as seen up to now). A dedicated `content` attribute carries the data.

Multiple specific classes are defined in orde to tell the LLM who is speaking:

- `SystemMessage`: system prompt.

- `HumanMessage`: user prompt.

- `AIMessage`: the assistant's, which can also contain structured tool_calls.

- `ToolMessage`: the output of an executed tool call.

In [ ]:
import json
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

MAX_DISCUSSION_TURNS = 4

def remove_chat_template_annotations(raw_response: str) -> str:
    if "<|start_header_id|>assistant<|end_header_id|>" in raw_response:
            raw_response = raw_response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
    return raw_response


# Pydantic object
class DiscussionTurn(BaseModel):
    message: str = Field(description="Proposal, critique, or question for your partner")
    ready_to_code: bool = Field(description="True only when both agree on the exact fix")
    agreed_approach: str = Field(default="", description="Concise summary of the agreed fix")

    @staticmethod
    def from_llm_response(raw_response: str, turn: int) -> "DiscussionTurn":
        raw_response = remove_chat_template_annotations(raw_response)

        match = re.search(r"\{.*\}", raw_response, re.DOTALL)
        data = json.loads(match.group(0) if match else raw_response)
        response = DiscussionTurn(**data)
        if turn == 0:
            response.ready_to_code = False
        return response



def discuss_node(state: PairState) -> dict:
    # ---------------- PROMPT SETUP -------------------
    turn = state["discussion_turns"]
    speaker = "Alice" if turn % 2 == 0 else "Bob"
    listener = "Bob" if turn % 2 == 0 else "Alice"
    role = "Driver" if speaker == state["driver"] else "Navigator"

    print(f"\n--- Node: discuss (Turn {turn + 1} | {speaker} [{role}] | Target Test #{state['current_test_idx']}) ---")


    sys_msg = SystemMessage(content=(""))  # TODO define a prompt to tell the LLM who it is (personas), the context (pair programming, test-driven), and the expected output format (DiscussionTurn object)

    turn_instruction = "" # TODO define a prompt to instruct the LLM on how to proceed in this discussion turn

    # ---------------- PROMPT SETUP -------------------


    # ------- TRY/EXCEPT to obtain a valid response ---
    messages_copy = [sys_msg, turn_instruction] + state["messages"][-6:] # Temp variable for try/except loops, so that the actual list of messages remains clean
    # IMPORTANT: "[-6:]" serves at considering only the last four messages and prevent OutOfMemoryError. You can reduce the number of messages from 6 to 4 if you have less GPU

    for attempt in range(3):
        raw = llm.invoke(messages_copy).content
        try:
            response = DiscussionTurn.from_llm_response(raw_response=raw, turn=turn)
            print(f"Attempt {attempt} succeeded | ready_to_code={response.ready_to_code} | agreed_approach='{response.agreed_approach}'")
            break

        except Exception as e:
            print(f"Attempt {attempt} failed ({type(e).__name__}: {e}). Trying to convert into a DiscussionTurn object this raw llm response:\n{raw}\n")

            # Self-refinement prompt
            messages_copy.append(HumanMessage(content="")) # TODO define a prompt to make the LLM output the information in the correct format
    else:
        # Fallback after 3 attempts
        print("All 3 attempts failed. Using fallback response to generate a DiscussionTurn object.")
        response = DiscussionTurn() # TODO define a DiscussionTurn object which can be used as a fallback, in order to continue the session

    # ------- TRY/EXCEPT to obtain a valid response ---

    return { # The node function returns the updates to the state variable
        "messages": [HumanMessage(content=f"[{speaker}]: {response.message}")], # ATTENTION: we only return the new messages. Since the `messages` attribute has been tagged as `Annotated` in PairState, each new value is appended in the history of states
        "discussion_turns": turn + 1,
        "ready_to_code": response.ready_to_code,
        "agreed_approach": response.agreed_approach if response.ready_to_code else state["agreed_approach"],
    }

### Step 4: LanGraph node for tests execution

First of all, we need to properly identify the current target test within the file containing the whole test suite.

For this purpose, we'll use the Python's built-in ast library, that lets you parse Python source code into an ***AST structure***.

---

An **AST (Abstract Syntax Tree)** is a tree-like representation of the syntactic structure of source code. Each node in the tree represents a construct occurring in the code (such as a function definition, loop, variable, or operator).

---

![AST_example.jpg)](AST_example.jpg)

In [ ]:
import ast
from pathlib import Path

TESTS_FILE_PATH = Path("test_cases_01.py")

def get_test_names(tests_file_path: Path) -> list[str]:
    """Extracts top-level test function names in file order."""
    tree = ast.parse(tests_file_path.read_text(encoding="utf-8"))
    return [
        # TODO loop over all the objects present in `tree.body`, and select only the ones of class `ast.FunctionDef` having the `node.name` starting with "test_"
    ]

ALL_TESTS = get_test_names(TESTS_FILE_PATH)

print(f"Loaded {len(ALL_TESTS)} tests")

Then, we can implement the LangGraph node responsible for test execution. As before we'll run pytest inside a subprocess.

Since we need to pass the tests results in an input prompt, we need to reduce their verbosity. Consequently, we'll run pytest with the following flags:

`--maxfail=3`: Halts execution after the first 3 test failures rather than running the entire suite. This prevents overwhelming smaller models (such as Llama-3B) with more error context than they can effectively process.

`--tb=short`: Outputs a concise traceback showing only the failing line.

`-q` (`--quiet`): Reduces console verbosity, hiding extra environment details and displaying minimal progress indicators.

`--no-header`: Suppresses the startup banner (Python version, pytest version, root directory, and active plugins).

`--color-no`: Color coding is not used to communicate information. Everything is written explicitly.

In [ ]:
import re
import subprocess
import sys

SOLUTION_FILE_PATH = Path("function_01.py")

def run_tests_node(state: PairState) -> dict:
    # TODO write down the code to be tested

    # -------- Identify the tests to be run --------
    target_idx = "" # TODO retrieve the index of the target test from the LangGraph's state
    tests_to_run = [] # TODO select the tests from the ALL_TESTS list. In order to properly perform regression testing, the selected tests should include all the test cases up to the current target one
    node_ids = [f"{TESTS_FILE_PATH.name}::{name}" for name in tests_to_run] # ids needed by pytest
    # -------- Identify the tests to be run --------

    # -------- Running the tests --------
    try:
        _result = subprocess.run(
            [
                sys.executable, "-m", "pytest",
                "--maxfail=3", "--tb=short", "-q", "--no-header", "--color=no",
                *node_ids,
            ],
            capture_output=True,
            text=True,
            timeout=30,
        )
    except subprocess.TimeoutExpired:
        timeout_msg = "Test execution timed out after 30s (possible infinite loop)."
        return {
            # TODO return the updates to the LangGraph state
        }

    output = (_result.stdout + "\n" + _result.stderr).strip()
    # -------- Running the tests --------


    # -------- HAPPY PATH --------
    if _result.returncode == 0:
        next_driver = "" # TODO assign the driver for the next coding step
        return {
            # TODO return the updates to the LangGraph state
        }
    # -------- HAPPY PATH --------


    # --- ERROR PATH ---
    target_test_name = ALL_TESTS[target_idx]

    if target_idx > 0 and f"FAILED {TESTS_FILE_PATH.name}::{target_test_name}" not in output:
        label = "REGRESSION"
    else:
        label = "TARGET TEST FAILED"

    # Extract traceback lines (skip the 'short test summary info' section to avoid duplication)
    lines = [line for line in output.splitlines() if line.strip()]
    summary_idx = next((i for i, l in enumerate(lines) if "short test summary info" in l), len(lines))
    traceback_details = "\n".join(lines[:summary_idx])

    return {
        # TODO return the updates to the LangGraph state
    }

### Step 5: LanGraph node for code editing

We define here:
 1. The code to allow the Driver agent to update the codebase;
2.  The code to eventually perform rollback to the last known valid state of the codebase.

Both nodes will produce an update to the LangGraph's global state.

In [ ]:
def driver_edit_node(state: PairState) -> dict:
    print(f"\n--- Node: driver_edit (Driver: {state['driver']}) ---")

    # ---------------- PROMPT SETUP -------------------
    prompt = (
        f"" # TODO define the prompt fo the agent with the "Driver" role. Instruct it to modify the code based on the approach agreed during the previous discussion
    )
    # ---------------- PROMPT SETUP -------------------


    # -------- INVOKE LLM and PARSE RESPONSE ----------
    updated_code = llm.invoke(prompt).content
    updated_code = remove_chat_template_annotations(updated_code)
    updated_code = updated_code.replace("```python", "").replace("```", "").strip()

    print(f"Generated code:\n```python\n{updated_code}\n```")
    # -------- INVOKE LLM and PARSE RESPONSE ----------

    return {
        # TODO return the updates to the LangGraph state
    }

def rollback_node(state: PairState) -> dict:
    return { # Code rollback + counters reset
        "code": state["last_known_good_code"],
        # TODO return the updates to the LangGraph state, including a message that communicates to the agents that a rollback has been performed
    }

### Step 6: Put everything together in the LangGraph's StateGraph

This will build up the cyclic graph represented in the figure above.

To route the flow through the arcs of the graph, we will use **Conditional Edges**. They are, basically, functions containing an if-else construct and returning the name of the next node.

The special value stored in the **END** constant is returned in order to direct the flow towards the end state and terminate the execution.

In [ ]:
from langgraph.graph import StateGraph, END

MAX_FAILED_ATTEMPTS = 3

# Node names
NODE_RUN_TESTS = "run_tests"
NODE_DISCUSS = "discuss"
NODE_DRIVER_EDIT = "driver_edit"
NODE_ROLLBACK = "rollback"

# ------------- ROUTE NODES ----------------------

def route_after_tests(state: PairState) -> str:
    # TODO write the code to:
    # - make the graph to terminate, in case all tests passed in the previous node
    # - perform rollback in case the maximum number of failed tests executions has been reached
    # - in case all tests passed and there are still tests to be considered in the suite, go to the discussion node
    pass


def route_after_discuss(state: PairState) -> str:
    # TODO write the code to decide whether to continue the discussion or to go to the code update node
    pass

# ------------- ROUTE NODES ----------------------



# ------------- LangGraph's graph ----------------

workflow = StateGraph(PairState)

# TODO for each node defined above (run_tests_node, discuss_node, driver_edit_node, rollback_node), call the function add_node() with the assigned name
workflow.add_node()

workflow.set_entry_point(NODE_RUN_TESTS)

# TODO for each route node defined above in this cell, call the add_conditional_edges() function. This is needed to tell LangGraph after which node the conditional edges are executed
workflow.add_conditional_edges()

# TODO call the function add_edge() to connect nodes that do not need a conditional edge in between
# HINT: use the image of the diagram provided above
workflow.add_edge()

app = workflow.compile()  # Transforms the declared graph into a runnable workflow

### Final step: run the graph

We need to create an initial state to be passed to the function `app.invoke()`. As initial code, we'll provide the function signature.

Moreover, we write some code to track the execution step-by-step.

In [ ]:
FUNCTION_SIGNATURE = (
"""def compute_average(grades: list[float]) -> float:
    \"\"\"
    Grades can be from 18 to 30, or 30Laude (which is represented by the int number 33).
    The average is computed excluding the best and worse grade.

    Args:
        grades (list): A list of six grades, where each grade is between 18 and 33.

    Returns:
        float: The computed average.
    \"\"\"
    pass
""")


initial_state: PairState = {
    # TODO define all the values for the initial state
}

config = {"recursion_limit": 500} # recursion_limit: Max number of node transitions before LangGraph aborts to prevent infinite loops (defaults to 25).


# Stream step-by-step
for event in app.stream(initial_state, config=config):
    for node_name, state_update in event.items():
        if node_name not in ("discuss", "driver_edit"):
            print(f"\n--- Node: {node_name} ---")
        if "messages" in state_update and state_update["messages"]:
            print(state_update["messages"][-1].content)


# In alternative, you can run everything and get the final state
# final_state = app.invoke(initial_state, config=config)
# print(final_state["code"])